# 评估驱动的系统设计：从原型到生产

## 概述

### 本 cookbook 的目的

本 cookbook 提供一份**实用的**端到端指南，介绍如何有效地将评估（evals）作为核心流程，构建一个生产级的自主系统，以取代劳动密集型的人工工作流。它直接源自我们在项目协作中积累的经验：在这些项目中，用户一开始往往既没有干净的标注数据，也没有对问题的完美理解——大多数教程都会略过这两个问题，但在实践中它们几乎总是严峻的挑战。

将评估作为核心流程，可以避免“戳一戳、碰碰运气”式的猜测和对准确率的主观印象判断，转而要求工程上的严谨。这意味着我们可以在成本权衡和投入方面做出有原则的决策。 

### 目标读者

本指南面向希望获得入门教程之外实用指导的 ML/AI 工程师和解决方案架构师。本 notebook 完全可执行，并尽可能模块化地组织，以便你直接在自己的应用中使用其中的代码示例。

### 主线叙事：从小小的种子到生产系统

我们将跟随一个贴近现实的故事线：取代用于核验报销的人工收据分析服务。

* **从小处着手：** 从一小组标注数据（零售收据）开始。许多企业并没有良好的真实标注（ground truth）数据集。 
* **逐步构建：** 开发一个最小可行系统，并建立初始评估。 
* **与业务对齐：** 结合业务 KPI 和金额影响来衡量评估表现，有针对性地投入精力，避免在低影响的改进上浪费时间。
* **评估驱动的迭代：** 利用评估分数推动模型改进，再用更好的模型处理更多数据，以扩展评估并发现更多可改进之处，如此迭代提升。

### 如何使用本 cookbook

本 cookbook 以评估为中心，贯穿构建 LLM 应用的整个生命周期。

1. 如果你主要对其中的思路感兴趣，可以通读文字，略读代码。
2. 如果你是因为手头的其他工作而来，可以直接跳到相关章节，深入研究那里的代码，复制并根据需要进行调整。
3. 如果你想真正理解这一切是如何运作的，请下载本 notebook，边读边运行单元格；修改代码做出自己的改动，检验你的假设，确保你真正理解各部分是如何协同工作的。

> 注意：如果你的 OpenAI 组织采用了零数据保留（ZDR）策略，Evals 仍然可用，但会为维护应用状态而保留数据。

## 用例：收据解析

为了精简本指南，我们将使用一个小型的假设问题，但它仍然足够复杂，值得进行细致且多维度的评估。特别是，我们将聚焦于如何在可用数据有限的情况下解决问题，因此我们使用的数据集相当小。

### 问题定义

在本指南中，我们假设起点是一个审核和归档收据的工作流。虽然总体而言，这个问题已有许多成熟的解决方案，但它与其他许多缺乏前人工作的问题类似；此外，即使存在良好的企业级解决方案，通常仍有需要人工投入时间的“最后一公里”问题。

在我们的场景中，假设有这样一个流水线：

* 人们上传收据照片
* 会计团队审核每张收据，对费用进行分类，并批准或审计该费用

根据对会计团队的访谈，他们做决策的依据是

1. 商户
2. 地理位置
3. 费用金额
4. 购买的商品或服务
5. 手写备注或批注

我们期望系统能在无需人工干预的情况下处理大多数收据，但会将低置信度的决策升级给人工 QA。我们将着重降低会计流程的总成本，它取决于

1. 原有/当前系统处理每张收据的运行成本
2. 新系统发送给 QA 的收据数量
3. 新系统处理每张收据的运行成本，加上所有固定成本
4. 错误带来的业务影响，包括被剔出待审核的收据以及被漏掉的错误
5. 开发和集成该系统的工程成本

### 数据集概览

收据图片来自 Roboflow 发布的、采用 CC by 4.0 许可的 [Receipt Handwriting Detection Computer Vision Project](https://universe.roboflow.com/newreceipts/receipt-handwriting-detection) 数据集。我们添加了自己的标签和叙事设定，以便用少量示例讲述一个完整的故事。

## 项目生命周期

并非每个项目都会以相同的方式推进，但项目通常具有一些共同的重要组成部分。

![项目生命周期](../../../images/partner_project_lifecycle.png)

实线箭头表示主要的推进过程或步骤，而虚线表示对问题的理解是持续进行的——对客户领域了解得越多，就越会影响流程的每一步。下面我们将详细考察其中几个迭代改进的循环。 
并非每个项目都会以相同的方式推进，但项目通常具有一些共同的重要组成部分。

### 1. 理解问题

通常，启动工程流程的决定是由领导层做出的，他们了解业务影响，但无需了解流程细节。在我们的示例中，我们要构建的系统旨在取代一个非 AI 的工作流。从某种意义上说，这是理想情况：我们有一批领域专家，即*当前正在执行该任务的人*，我们可以访谈他们以了解任务细节，并依靠他们帮助制定合适的评估。

这一步并不会在我们开始构建系统之前结束；我们最初的评估总是对问题空间不完整的理解，随着我们逐渐接近解决方案，我们会不断深化这种理解。

### 2. 收集示例（采集数据）

真实世界的项目很少在一开始就拥有获得满意解决方案所需的全部数据，更不用说建立信心了。

在我们的场景中，假设我们有相当数量的系统*输入*样本（即收据图片），但起步时没有任何完整标注的数据。我们发现，在将现有流程自动化时，这种情况并不少见。我们将演示如何与领域专家协作，逐步扩展测试集和训练集，并让评估逐渐变得更加全面。

### 3. 构建端到端的 V0 系统

我们希望尽快搭建出系统的骨架。我们不需要一个表现良好的系统——只需要一个能接收正确输入并提供正确类型输出的东西。通常，这几乎就像在提示词中描述任务、加入输入，并使用单个模型（通常配合结构化输出）做出初步的尽力尝试一样简单。

### 4. 标注数据并构建初始评估

我们发现，在缺少既定真实标注的情况下，使用系统的早期版本生成“草稿”真实数据、再由领域专家进行标注或纠正的做法并不少见。

一旦构建好端到端系统，我们就可以开始处理现有的输入，生成看似合理的输出。我们会把这些输出发送给领域专家进行评分和纠正。我们将利用这些纠正结果，以及关于专家如何做出决策的交流，来设计更多评估，并将专业知识融入系统。

### 5. 将评估映射到业务指标

在着手纠正每一个错误之前，我们需要确保时间投入是有效的。这一阶段最关键的任务是审视我们的评估，理解它们与关键目标之间的联系。

- 退后一步，评估系统潜在的成本和收益
- 找出哪些评估指标直接反映这些成本和收益
- 例如，某项评估“失败”的代价是什么？我们衡量的东西是否有价值？
- 构建一个（非 LLM 的）模型，用评估指标给出金额价值
- 在性能（准确率或速度）与开发和运行成本之间取得平衡

### 6. 逐步改进系统和评估

确定了哪些工作最值得投入之后，我们就可以开始迭代改进系统。评估充当客观的指引，让我们知道系统何时已经足够好，并确保我们能避免或识别出回归。 

### 7. 集成 QA 流程并持续改进

评估不仅仅用于开发阶段。对生产服务的全部或部分进行插桩，随着时间推移会发现更多有用的测试和训练样本，识别出错误的假设，或找到覆盖不足的领域。这也是确保你的模型在初始开发完成很久之后依然表现良好的唯一方法。

## V0 系统构建

在实践中，我们可能会构建一个通过 REST API 运行的系统，或许还带有某种 Web 前端，可以访问一组组件和资源。为了本 cookbook 的目的，我们将其提炼为一对函数：`extract_receipt_details` 和 `evaluate_receipt_for_audit`，它们共同决定如何处理给定的收据。

- `extract_receipt_details` 接收一张图片作为输入，生成包含收据重要细节的结构化输出。
- `evaluate_receipt_for_audit` 接收该结构作为输入，决定是否应对该收据进行审计。

> 像这样将流程拆分为多个步骤有利也有弊；如果流程由小而独立的步骤组成，就更容易检查和开发。但你可能会逐步丢失信息，实际上就像让你的智能体玩“传话游戏”。在本 notebook 中，我们拆分了步骤，并且不让审计员看到实际的收据，因为这对我们想讨论的评估更有启发意义。

我们从第一步开始，即逐字的数据提取。这是*中间*数据：它是人们会隐式查看、但通常不会被记录下来的信息。也正因如此，我们往往没有可以使用的标注数据。

In [ ]:
%pip install --upgrade openai pydantic python-dotenv rich persist-cache -qqq
%load_ext dotenv
%dotenv

# Place your API key in a file called .env
# OPENAI_API_KEY=sk-...

### 结构化输出模型

用结构化输出捕获有意义的信息。

In [ ]:
from pydantic import BaseModel


class Location(BaseModel):
    city: str | None
    state: str | None
    zipcode: str | None


class LineItem(BaseModel):
    description: str | None
    product_code: str | None
    category: str | None
    item_price: str | None
    sale_price: str | None
    quantity: str | None
    total: str | None


class ReceiptDetails(BaseModel):
    merchant: str | None
    location: Location
    time: str | None
    items: list[LineItem]
    subtotal: str | None
    tax: str | None
    total: str | None
    handwritten_notes: list[str]

> *注意*：通常我们会对上面的数字使用 `decimal.Decimal` 对象，对 `time` 字段使用 `datetime.datetime` 对象，但这两者都不能很好地反序列化。在本 cookbook 中，我们将使用字符串，但在实践中，你需要再加一层转换，才能得到经过正确验证的输出。

### 基本信息提取

我们来构建 `extract_receipt_details` 函数。

通常，在第一次尝试某个可能可行的方案时，我们只需把目前收集到的文档提供给 ChatGPT，让它生成一个提示词。在拥有可以用来给自己打分的基准之前，不值得在提示词工程上花太多时间！下面是 o4-mini 根据上述问题描述生成的提示词。

In [ ]:
BASIC_PROMPT = """
Given an image of a retail receipt, extract all relevant information and format it as a structured response.

# Task Description

Carefully examine the receipt image and identify the following key information:

1. Merchant name and any relevant store identification
2. Location information (city, state, ZIP code)
3. Date and time of purchase
4. All purchased items with their:
   * Item description/name
   * Item code/SKU (if present)
   * Category (infer from context if not explicit)
   * Regular price per item (if available)
   * Sale price per item (if discounted)
   * Quantity purchased
   * Total price for the line item
5. Financial summary:
   * Subtotal before tax
   * Tax amount
   * Final total
6. Any handwritten notes or annotations on the receipt (list each separately)

## Important Guidelines

* If information is unclear or missing, return null for that field
* Format dates as ISO format (YYYY-MM-DDTHH:MM:SS)
* Format all monetary values as decimal numbers
* Distinguish between printed text and handwritten notes
* Be precise with amounts and totals
* For ambiguous items, use your best judgment based on context

Your response should be structured and complete, capturing all available information
from the receipt.
"""

In [ ]:
import base64
import mimetypes
from pathlib import Path

from openai import AsyncOpenAI

client = AsyncOpenAI()


async def extract_receipt_details(
    image_path: str, model: str = "o4-mini"
) -> ReceiptDetails:
    """Extract structured details from a receipt image."""
    # Determine image type for data URI.
    mime_type, _ = mimetypes.guess_type(image_path)

    # Read and base64 encode the image.
    b64_image = base64.b64encode(Path(image_path).read_bytes()).decode("utf-8")
    image_data_url = f"data:{mime_type};base64,{b64_image}"

    response = await client.responses.parse(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": BASIC_PROMPT},
                    {"type": "input_image", "image_url": image_data_url},
                ],
            }
        ],
        text_format=ReceiptDetails,
    )

    return response.output_parsed

### 在一张收据上测试

我们只评估一张收据并人工检查，看看一个使用朴素提示词的智能模型能做到什么程度。

<img src="../../../images/Supplies_20240322_220858_Raven_Scan_3_jpeg.rf.50852940734939c8838819d7795e1756.jpg" alt="Walmart_image" width="400"/>

In [ ]:
from rich import print

receipt_image_dir = Path("data/test")
ground_truth_dir = Path("data/ground_truth")

example_receipt = Path(
    "data/train/Supplies_20240322_220858_Raven_Scan_3_jpeg.rf.50852940734939c8838819d7795e1756.jpg"
)
result = await extract_receipt_details(example_receipt)

如果重新运行，我们会得到不同的答案，但它通常能把大部分内容弄对，只有少量错误。下面是一个具体示例：

In [ ]:
walmart_receipt = ReceiptDetails(
    merchant="Walmart",
    location=Location(city="Vista", state="CA", zipcode="92083"),
    time="2023-06-30T16:40:45",
    items=[
        LineItem(
            description="SPRAY 90",
            product_code="001920056201",
            category=None,
            item_price=None,
            sale_price=None,
            quantity="2",
            total="28.28",
        ),
        LineItem(
            description="LINT ROLLER 70",
            product_code="007098200355",
            category=None,
            item_price=None,
            sale_price=None,
            quantity="1",
            total="6.67",
        ),
        LineItem(
            description="SCRUBBER",
            product_code="003444193232",
            category=None,
            item_price=None,
            sale_price=None,
            quantity="2",
            total="12.70",
        ),
        LineItem(
            description="FLOUR SACK 10",
            product_code="003444194263",
            category=None,
            item_price=None,
            sale_price=None,
            quantity="1",
            total="0.77",
        ),
    ],
    subtotal="50.77",
    tax="4.19",
    total="54.96",
    handwritten_notes=[],
)

模型正确提取了很多内容，但重命名了部分明细项——而且实际上改错了。更重要的是，它把一些价格弄错了，并且决定不对任何明细项进行分类。

没关系，我们并不指望在这个阶段就得到完美的答案！我们的目标是构建一个可以评估的基础系统。这样，当我们开始迭代时，就不会凭“感觉”去追求一个*看起来*更好的东西——而是在以工程化的方式构建可靠的解决方案。不过首先，我们要加上一个操作决策，来完成我们的草稿系统。

### 操作决策

接下来，我们需要形成闭环，基于收据得出实际的决策。这部分看起来非常相似，所以我们直接给出代码，不作注释。

通常，第一轮会从能力最强的模型开始——目前是 `o3`——在确立正确性之后，再尝试不同的模型，分析各种权衡对业务的影响，并可能考虑这些问题能否通过迭代加以弥补。客户可能愿意为了更低的延迟或成本而接受一定的准确率损失，或者调整架构可能更有效地实现成本、延迟和准确率目标。稍后我们将介绍如何明确、客观地进行这些权衡。 

对于本 cookbook 来说，`o3` 可能太好了。我们将在第一轮中使用 `o4-mini`，这样会出现一些推理错误，可以用来演示出现这类错误时的应对方法。

接下来，我们需要形成闭环，基于收据得出实际的决策。这部分看起来非常相似，所以我们直接给出代码，不作注释。

In [ ]:
from pydantic import BaseModel, Field

audit_prompt = """
Evaluate this receipt data to determine if it need to be audited based on the following
criteria:

1. NOT_TRAVEL_RELATED:
   - IMPORTANT: For this criterion, travel-related expenses include but are not limited
   to: gas, hotel, airfare, or car rental.
   - If the receipt IS for a travel-related expense, set this to FALSE.
   - If the receipt is NOT for a travel-related expense (like office supplies), set this
   to TRUE.
   - In other words, if the receipt shows FUEL/GAS, this would be FALSE because gas IS
   travel-related.

2. AMOUNT_OVER_LIMIT: The total amount exceeds $50

3. MATH_ERROR: The math for computing the total doesn't add up (line items don't sum to
   total)

4. HANDWRITTEN_X: There is an "X" in the handwritten notes

For each criterion, determine if it is violated (true) or not (false). Provide your
reasoning for each decision, and make a final determination on whether the receipt needs
auditing. A receipt needs auditing if ANY of the criteria are violated.

Return a structured response with your evaluation.
"""


class AuditDecision(BaseModel):
    not_travel_related: bool = Field(
        description="True if the receipt is not travel-related"
    )
    amount_over_limit: bool = Field(description="True if the total amount exceeds $50")
    math_error: bool = Field(description="True if there are math errors in the receipt")
    handwritten_x: bool = Field(
        description="True if there is an 'X' in the handwritten notes"
    )
    reasoning: str = Field(description="Explanation for the audit decision")
    needs_audit: bool = Field(
        description="Final determination if receipt needs auditing"
    )


async def evaluate_receipt_for_audit(
    receipt_details: ReceiptDetails, model: str = "o4-mini"
) -> AuditDecision:
    """Determine if a receipt needs to be audited based on defined criteria."""
    # Convert receipt details to JSON for the prompt
    receipt_json = receipt_details.model_dump_json(indent=2)

    response = await client.responses.parse(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": audit_prompt},
                    {"type": "input_text", "text": f"Receipt details:\n{receipt_json}"},
                ],
            }
        ],
        text_format=AuditDecision,
    )

    return response.output_parsed

整个流程的示意图显示了两次 LLM 调用：

![流程图](../../../images/partner_process_flowchart.png)

如果用这个模型处理上面的示例，会得到如下结果——同样，这里使用的是一个示例结果。你运行代码时可能会得到略有不同的结果。

In [ ]:
audit_decision = await evaluate_receipt_for_audit(result)
print(audit_decision)

In [ ]:
audit_decision = AuditDecision(
    not_travel_related=True,
    amount_over_limit=True,
    math_error=False,
    handwritten_x=False,
    reasoning="""
    The receipt from Walmart is for office supplies, which are not travel-related, thus NOT_TRAVEL_RELATED is TRUE.
    The total amount of the receipt is $54.96, which exceeds the limit of $50, making AMOUNT_OVER_LIMIT TRUE.
    The subtotal ($50.77) plus tax ($4.19) correctly sums to the total ($54.96), so there is no MATH_ERROR.
    There are no handwritten notes, so HANDWRITTEN_X is FALSE.
    Since two criteria (amount over limit and travel-related) are violated, the receipt
    needs auditing.
    """,
    needs_audit=True,
)

这个示例说明了为什么我们关心端到端评估，以及为什么不能孤立地使用它们。在这里，初始提取存在 OCR 错误，传给审计员的价格加起来与总额不符，但审计员没有发现这一点，并断言不存在计算错误。不过，漏掉这一点并没有改变审计决策，因为它确实发现了该收据需要审计的另外两个原因。

因此，`AuditDecision` 在事实上是不正确的，但我们关心的决策是正确的。这为我们提供了一个改进点，同时也指导我们在何处、何时投入工程精力方面做出合理的选择。

话虽如此，我们来构建一些评估吧！

## 初始评估

一旦有了一个具备最低功能的系统，我们就应该处理更多输入，并请领域专家帮助制作真实标注数据。从事专业工作的领域专家可能没有多少时间投入我们的项目，因此我们要讲求效率、从小处着手，起初追求广度而非深度。

> 如果你的数据*不*需要领域专业知识，那么你应该使用标注解决方案（例如 [Label Studio](https://labelstud.io/)），并在政策、预算和数据可用性限制允许的范围内尽可能多地标注数据。
> 在本例中，我们将假设数据标注是一种稀缺资源；我们每周可以依赖少量标注，但这些人还有其他工作职责，他们的时间和协助意愿可能有限。与这些专家坐在一起帮助标注示例，可以让后续示例的选择更加高效。

由于我们的链路包含两个步骤，我们将收集类型为 `[FilePath, ReceiptDetails, AuditDecision]` 的元组。通常的做法是取未标注的样本，用我们的模型处理它们，然后让专家纠正输出。在本 notebook 中，我们已经对 `data/test` 中的所有收据图片完成了这一过程。

### 其他考虑因素

不过，事情还不止于此，因为在评估多步骤流程时，既要了解端到端的表现，也要了解每个单独步骤的表现，*并以前一步骤的输出为条件*。

在本例中，我们希望评估：

1. 给定一张输入图片，我们提取所需信息的效果如何？
2. 给定收据信息，我们在审计决策上的**判断**有多好？
3. 给定一张输入图片，我们在做出最终审计决策方面有多**成功**？

第 2 点和第 3 点措辞不同，是因为如果我们给审计员提供了错误的数据，就预期它会得出错误的结论。我们*希望*的是，即使证据具有误导性，也能确信审计员是基于可用证据做出了正确的决策。如果不关注这种情况，我们最终可能会训练审计员忽略其输入，导致整体表现下降。

### 评分器

评估的核心组成部分是[评分器（grader）](https://platform.openai.com/docs/guides/graders)。我们最终的评估将使用 18 个评分器，但只用了三种类型，而且它们在概念上都相当简单。

下面分别是一个字符串检查评分器、一个文本相似度评分器以及一个模型评分器的示例。

In [ ]:
example_graders = [
    {
        "name": "Total Amount Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_receipt_details.total }}",
        "reference": "{{ item.correct_receipt_details.total }}",
    },
    {
        "name": "Merchant Name Accuracy",
        "type": "text_similarity",
        "input": "{{ item.predicted_receipt_details.merchant }}",
        "reference": "{{ item.correct_receipt_details.merchant }}",
        "pass_threshold": 0.8,
        "evaluation_metric": "bleu",
    },
]

# A model grader needs a prompt to instruct it in what it should be scoring.
missed_items_grader_prompt = """
Your task is to evaluate the correctness of a receipt extraction model.

The following items are the actual (correct) line items from a specific receipt.

{{ item.correct_receipt_details.items }}

The following items are the line items extracted by the model.

{{ item.predicted_receipt_details.items }}

Score 0 if the sample evaluation missed any items from the receipt; otherwise score 1.

The line items are permitted to have small differences or extraction mistakes, but each
item from the actual receipt must be present in some form in the model's output. Only
evaluate whether there are MISSED items; ignore other mistakes or extra items.
"""

example_graders.append(
    {
        "name": "Missed Line Items",
        "type": "score_model",
        "model": "o4-mini",
        "input": [{"role": "system", "content": missed_items_grader_prompt}],
        "range": [0, 1],
        "pass_threshold": 1,
    }
)

每个评分器评估预测输出的某一部分。这可能是针对结构化输出中某个特定字段的非常狭窄的检查，也可能是对整个输出进行评判的更全面的检查。有些评分器无需上下文，可以孤立地评估输出（例如，一个评判某段文字是否粗鲁或不当的 LLM 评判者）。另一些可以基于输入和输出进行评估，而我们这里使用的评分器则依赖于输出以及用于比较的真实（正确）输出。

使用 Evals 最直接的方式是提供一个提示词和一个模型，让评估在输入上运行并自行生成输出。另一种有用的方法是使用之前记录的响应或补全作为输出来源。虽然没有那么简单，但最灵活的做法是提供一个包含我们希望它使用的全部内容的条目——这样，“预测”函数就可以是任意系统，而不局限于单次模型调用。下面的示例正是这样使用的；下面所示的 `EvaluationRecord` 将用于填充 `{{ }}` 模板变量。

> **关于模型选择的说明：**  
> 选择合适的模型至关重要。虽然在生产中通常更倾向于使用更快、更便宜的模型，但在开发工作流中优先使用能力最强的模型会更有益。在本指南中，我们在系统任务和基于 LLM 的评分中都使用 `o4-mini`——虽然 `o3` 能力更强，但根据我们的经验，与成本的大幅增加相比，输出质量的差异并不大。在实践中，每位工程师每天在评估上花费 $10+ 是很常见的，但扩大到每位工程师每天 $100+ 可能难以为继。
>
> 尽管如此，定期使用 `o3` 这样更先进的模型进行基准测试仍然很有价值。如果你观察到显著的改进，可以考虑在具有代表性的一部分评估数据上采用它。不同模型之间的差异可以揭示重要的边界情况，并指导系统改进。

In [ ]:
import asyncio


class EvaluationRecord(BaseModel):
    """Holds both the correct (ground truth) and predicted audit decisions."""

    receipt_image_path: str
    correct_receipt_details: ReceiptDetails
    predicted_receipt_details: ReceiptDetails
    correct_audit_decision: AuditDecision
    predicted_audit_decision: AuditDecision


async def create_evaluation_record(image_path: Path, model: str) -> EvaluationRecord:
    """Create a ground truth record for a receipt image."""
    extraction_path = ground_truth_dir / "extraction" / f"{image_path.stem}.json"
    correct_details = ReceiptDetails.model_validate_json(extraction_path.read_text())
    predicted_details = await extract_receipt_details(image_path, model)

    audit_path = ground_truth_dir / "audit_results" / f"{image_path.stem}.json"
    correct_audit = AuditDecision.model_validate_json(audit_path.read_text())
    predicted_audit = await evaluate_receipt_for_audit(predicted_details, model)

    return EvaluationRecord(
        receipt_image_path=image_path.name,
        correct_receipt_details=correct_details,
        predicted_receipt_details=predicted_details,
        correct_audit_decision=correct_audit,
        predicted_audit_decision=predicted_audit,
    )


async def create_dataset_content(
    receipt_image_dir: Path, model: str = "o4-mini"
) -> list[dict]:
    # Assemble paired samples of ground truth data and predicted results. You could
    # instead upload this data as a file and pass a file id when you run the eval.
    tasks = [
        create_evaluation_record(image_path, model)
        for image_path in receipt_image_dir.glob("*.jpg")
    ]
    return [{"item": record.model_dump()} for record in await asyncio.gather(*tasks)]


file_content = await create_dataset_content(receipt_image_dir)

有了评分器和数据之后，创建和运行评估就非常简单了：

In [ ]:
from persist_cache import cache


# We're caching the output so that if we re-run this cell we don't create a new eval.
@cache
async def create_eval(name: str, graders: list[dict]):
    eval_cfg = await client.evals.create(
        name=name,
        data_source_config={
            "type": "custom",
            "item_schema": EvaluationRecord.model_json_schema(),
            "include_sample_schema": False,  # Don't generate new completions.
        },
        testing_criteria=graders,
    )
    print(f"Created new eval: {eval_cfg.id}")
    return eval_cfg


initial_eval = await create_eval(
    "Initial Receipt Processing Evaluation", example_graders
)

# Run the eval.
eval_run = await client.evals.runs.create(
    name="initial-receipt-processing-run",
    eval_id=initial_eval.id,
    data_source={
        "type": "jsonl",
        "source": {"type": "file_content", "content": file_content},
    },
)
print(f"Evaluation run created: {eval_run.id}")
print(f"View results at: {eval_run.report_url}")

运行该评估后，你就可以在 UI 中查看它，应该会看到类似下面的内容。 

（注意，如果你签署了零数据保留协议，这些数据不会被 OpenAI 存储，因此在此界面中将不可用。）
如下：

![摘要 UI](../../../images/partner_summary_ui.png)

你可以深入数据选项卡查看单个示例：

![详情 UI](../../../images/partner_details_ui.png)

## 将评估与业务指标关联起来

评估向你展示可以改进的地方，并帮助跟踪随时间推移的进展和回归。但上面这三个评估只是度量——我们需要赋予它们存在的意义（raison d'être）。

我们首先需要为收据处理的最后阶段添加评估，这样才能开始看到审计决策的结果。接下来我们需要的、也是最重要的，是一个*业务相关性模型*。

### 业务模型

根据新系统的表现来推算它可能带来的成本和收益，几乎从来都不容易。人们常常会回避给事情定量，因为他们知道其中存在多大的不确定性，也不想做出让自己难堪的猜测。这没关系；我们只需做出最佳猜测，如果以后获得更多信息，再完善模型即可。

在本 cookbook 中，我们将建立一个简单的成本结构：

- 我们公司每年处理 100 万张收据，基准成本为每张收据 $0.20
- 审计一张收据大约花费 $2
- 未能审计本应审计的收据，平均损失 $30
- 5% 的收据需要审计
- 现有流程
  - 有 97% 的时候能识别出需要审计的收据
  - 有 2% 的时候会把不需要审计的收据误判为需要审计

这为我们提供了两个基准比较：

- 如果我们正确识别每一张收据，将在审计上花费 $100,000
- 我们当前的流程在审计上花费 $135,000，并因未审计的费用损失 $45,000

此外，人工驱动的流程还要额外花费 $200,000。

我们预期服务能通过更低的运行成本来省钱（如果使用上面的提示词配合 `o4-mini`，约为每张收据 1 美分），但在审计和漏审方面是省钱还是亏钱，取决于系统的表现。将其写成一个简单的函数可能会很有用——下面编写的版本包含了上述因素，但忽略了细节，也没有考虑开发、维护和服务成本。

In [ ]:
def calculate_costs(fp_rate: float, fn_rate: float, per_receipt_cost: float):
    audit_cost = 2
    missed_audit_cost = 30
    receipt_count = 1e6
    audit_fraction = 0.05

    needs_audit_count = receipt_count * audit_fraction
    no_needs_audit_count = receipt_count - needs_audit_count

    missed_audits = needs_audit_count * fn_rate
    total_audits = needs_audit_count * (1 - fn_rate) + no_needs_audit_count * fp_rate

    audit_cost = total_audits * audit_cost
    missed_audit_cost = missed_audits * missed_audit_cost
    processing_cost = receipt_count * per_receipt_cost

    return audit_cost + missed_audit_cost + processing_cost


perfect_system_cost = calculate_costs(0, 0, 0)
current_system_cost = calculate_costs(0.02, 0.03, 0.20)

print(f"Current system cost: ${current_system_cost:,.0f}")


### 与评估重新关联

上述模型的意义在于，它让我们能够为原本只是一个数字的评估赋予含义。例如，在运行上面的系统时，我们在商户名称上有 85% 的时候是错的。但深入研究后发现，大多数情况是大小写问题，或者是“Shell Gasoline”与“Shell Oil #2144”之类的差异——顺着追查下去，这些问题似乎并不影响我们的审计决策，也不会改变我们的基本成本。

另一方面，我们似乎有大约一半的时候没能识别出收据上手写的“X”，而在漏掉收据上“X”的情况中，又有大约一半导致本应审计的收据没有被审计。这类情况在我们的数据集中占比偏高，但即便它们只占收据的 1%，这 50% 的失败率每年也会让我们损失 $75,000。

同样，我们似乎存在 OCR 错误，导致我们经常因为金额对不上而审计收据，比例高达 20%。这可能让我们损失将近 $400,000！

现在，我们可以添加更多评分器，并从审计决策的准确率倒推，确定应该重点关注哪些问题。

下面是我们其余的评分器，以及使用初始未优化提示词得到的结果。注意，此时我们的表现相当糟糕！在 20 个样本（8 个正例、12 个反例）中，我们出现了两个假阴性和两个假阳性。如果推算到整个业务，我们将因漏审损失 $375,000，并因不必要的审计损失 $475,000。

In [ ]:
simple_extraction_graders = [
    {
        "name": "Merchant Name Accuracy",
        "type": "text_similarity",
        "input": "{{ item.predicted_receipt_details.merchant }}",
        "reference": "{{ item.correct_receipt_details.merchant }}",
        "pass_threshold": 0.8,
        "evaluation_metric": "bleu",
    },
    {
        "name": "Location City Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_receipt_details.location.city }}",
        "reference": "{{ item.correct_receipt_details.location.city }}",
    },
    {
        "name": "Location State Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_receipt_details.location.state }}",
        "reference": "{{ item.correct_receipt_details.location.state }}",
    },
    {
        "name": "Location Zipcode Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_receipt_details.location.zipcode }}",
        "reference": "{{ item.correct_receipt_details.location.zipcode }}",
    },
    {
        "name": "Time Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_receipt_details.time }}",
        "reference": "{{ item.correct_receipt_details.time }}",
    },
    {
        "name": "Subtotal Amount Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_receipt_details.subtotal }}",
        "reference": "{{ item.correct_receipt_details.subtotal }}",
    },
    {
        "name": "Tax Amount Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_receipt_details.tax }}",
        "reference": "{{ item.correct_receipt_details.tax }}",
    },
    {
        "name": "Total Amount Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_receipt_details.total }}",
        "reference": "{{ item.correct_receipt_details.total }}",
    },
    {
        "name": "Handwritten Notes Accuracy",
        "type": "text_similarity",
        "input": "{{ item.predicted_receipt_details.handwritten_notes }}",
        "reference": "{{ item.correct_receipt_details.handwritten_notes }}",
        "pass_threshold": 0.8,
        "evaluation_metric": "fuzzy_match",
    },
]

item_extraction_base = """
Your task is to evaluate the correctness of a receipt extraction model.

The following items are the actual (correct) line items from a specific receipt.

{{ item.correct_receipt_details.items }}

The following items are the line items extracted by the model.

{{ item.predicted_receipt_details.items }}
"""

missed_items_instructions = """
Score 0 if the sample evaluation missed any items from the receipt; otherwise score 1.

The line items are permitted to have small differences or extraction mistakes, but each
item from the actual receipt must be present in some form in the model's output. Only
evaluate whether there are MISSED items; ignore other mistakes or extra items.
"""

extra_items_instructions = """
Score 0 if the sample evaluation extracted any extra items from the receipt; otherwise
score 1.

The line items are permitted to have small differences or extraction mistakes, but each
item from the actual receipt must be present in some form in the model's output. Only
evaluate whether there are EXTRA items; ignore other mistakes or missed items.
"""

item_mistakes_instructions = """
Score 0 to 10 based on the number and severity of mistakes in the line items.

A score of 10 means that the two lists are perfectly identical.

Remove 1 point for each minor mistake (typos, capitalization, category name
differences), and up to 3 points for significant mistakes (incorrect quantity, price, or
total, or categories that are not at all similar).
"""

item_extraction_graders = [
    {
        "name": "Missed Line Items",
        "type": "score_model",
        "model": "o4-mini",
        "input": [
            {
                "role": "system",
                "content": item_extraction_base + missed_items_instructions,
            }
        ],
        "range": [0, 1],
        "pass_threshold": 1,
    },
    {
        "name": "Extra Line Items",
        "type": "score_model",
        "model": "o4-mini",
        "input": [
            {
                "role": "system",
                "content": item_extraction_base + extra_items_instructions,
            }
        ],
        "range": [0, 1],
        "pass_threshold": 1,
    },
    {
        "name": "Item Mistakes",
        "type": "score_model",
        "model": "o4-mini",
        "input": [
            {
                "role": "system",
                "content": item_extraction_base + item_mistakes_instructions,
            }
        ],
        "range": [0, 10],
        "pass_threshold": 8,
    },
]


simple_audit_graders = [
    {
        "name": "Not Travel Related Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_audit_decision.not_travel_related }}",
        "reference": "{{ item.correct_audit_decision.not_travel_related }}",
    },
    {
        "name": "Amount Over Limit Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_audit_decision.amount_over_limit }}",
        "reference": "{{ item.correct_audit_decision.amount_over_limit }}",
    },
    {
        "name": "Math Error Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_audit_decision.math_error }}",
        "reference": "{{ item.correct_audit_decision.math_error }}",
    },
    {
        "name": "Handwritten X Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_audit_decision.handwritten_x }}",
        "reference": "{{ item.correct_audit_decision.handwritten_x }}",
    },
    {
        "name": "Needs Audit Accuracy",
        "type": "string_check",
        "operation": "eq",
        "input": "{{ item.predicted_audit_decision.needs_audit }}",
        "reference": "{{ item.correct_audit_decision.needs_audit }}",
    },
]


reasoning_eval_prompt = """
Your task is to evaluate the quality of *reasoning* for audit decisions on receipts.
Here are the rules for audit decisions:

Expenses should be audited if they violate any of the following criteria:
1. Expenses must be travel-related
2. Expenses must not exceed $50
3. All math should be correct; the line items plus tax should equal the total
4. There must not be an "X" in the handwritten notes

If ANY of those criteria are violated, the expense should be audited.

Here is the input to the grader:
{{ item.predicted_receipt_details }}

Below is the output of an authoritative grader making a decision about whether or not to
audit an expense. This is a correct reference decision.

GROUND TRUTH:
{{ item.correct_audit_decision }}


Here is the output of the model we are evaluating:

MODEL GENERATED:
{{ item.predicted_audit_decision }}


Evaluate:
1. For each of the 4 criteria, did the model correctly score it as TRUE or FALSE?
2. Based on the model's *scoring* of the criteria (regardless if it scored it
   correctly), did the model reason appropriately about the criteria (i.e. did it
   understand and apply the prompt correctly)?
3. Is the model's reasoning logically sound, sufficient, and comprehensible?
4. Is the model's reasoning concise, without extraneous details?
5. Is the final decision to audit or not audit correct?

Grade the model with the following rubric:
- (1) point for each of the 4 criteria that the model scored correctly
- (3) points for each aspect of the model's reasoning that is meets the criteria
- (3) points for the model's final decision to audit or not audit

The total score is the sum of the points, and should be between 0 and 10 inclusive.
"""


model_judgement_graders = [
    {
        "name": "Audit Reasoning Quality",
        "type": "score_model",
        "model": "o4-mini",
        "input": [{"role": "system", "content": reasoning_eval_prompt}],
        "range": [0, 10],
        "pass_threshold": 8,
    },
]

full_eval = await create_eval(
    "Full Receipt Processing Evaluation",
    simple_extraction_graders
    + item_extraction_graders
    + simple_audit_graders
    + model_judgement_graders,
)

eval_run = await client.evals.runs.create(
    name="complete-receipt-processing-run",
    eval_id=full_eval.id,
    data_source={
        "type": "jsonl",
        "source": {"type": "file_content", "content": file_content},
    },
)

eval_run.report_url

![大型摘要 UI](../../../images/partner_large_summary_ui.png)

## 让飞轮转起来

有了业务模型，就相当于有了一张地图，告诉我们哪些事值得做、哪些不值得做。我们的初始评估就像路标，让我们知道自己正朝着正确的方向前进；但最终我们还需要更多路标。在流程的这个阶段，我们通常有很多不同的事情可以做，其中有几个相互关联的循环，在一个循环上的改进会为另一个循环打开更多改进空间。

![开发飞轮](../../../images/partner_development_flywheel.png)

1. 评估向我们展示了可以改进的地方，我们可以立即用它们来指导模型选择、提示词工程、工具使用和微调策略。
2. 当系统在评估中表现良好时，我们的工作并没有结束。这时该*改进我们的评估*了。我们将处理更多数据，交给领域专家审核，并将纠正结果用于构建更好、更全面的评估。

这个循环可以持续一段时间。我们可以通过找出值得检查的“有趣”数据的有效前沿来加快这一进程。有几种技术可以做到这一点，一种简单的方法是对输入重复运行模型，优先标注那些答案不一致的输入。这在使用不同底层模型时效果尤其好，甚至使用能力较弱的模型往往也有帮助（如果一个笨模型和一个聪明的模型意见一致，那么这很可能不是一个难题）。

一旦在性能上似乎已经达到收益递减的阶段，我们就可以继续使用同样的技术来优化模型成本；如果我们有一个表现相当好的系统，那么微调或某种形式的模型蒸馏很可能让我们用更小、更便宜、更快的模型获得类似的性能。

## 系统改进

评估已经就位，并且我们了解了它们与业务指标之间的联系，终于可以把注意力转向改进系统的输出了。

上面我们提到，商户名称有 85% 的时候是错的，比我们评估的任何其他输出都多。这看起来相当糟糕，而且很可能只需少量工作就能大幅改进，但我们不妨从业务指标这一终点出发，倒推看看是哪些问题导致了错误的决策。

这样做之后，我们发现商户名称上的错误与最终的审计决策完全不相关，也没有证据表明它们对该决策有任何影响。根据我们的业务模型，实际上并没有改进它的必要——换句话说，*并非所有评估都重要*。相反，我们可以专门检查那些审计决策出错的示例。这样的示例只有两个（共 20 个）。仔细检查后，我们发现两种情况的问题都出在流水线的第二阶段：在提取结果没有问题的情况下做出了错误的决策。事实上，两者都源于未能对差旅相关费用进行正确推理。

在第一种情况中，购买的是一家汽车配件店的除雪刷。这有点属于边界情况，但我们的领域专家认定这是合理的差旅费用（因为司机可能需要用它清理挡风玻璃）。看起来，更详细地解释决策过程并提供一个类似的示例就能纠正这个错误。

在第二种情况中，购买的是一家家装店的一些工具。这些工具与正常驾驶毫无关系，因此这张收据应作为“非差旅相关费用”进行审计。在这种情况下，我们的模型*正确地*将其识别为非差旅相关的费用，但随后对这一事实的推理出了错，显然没有理解 `not_travel_related` 为 `true` 应意味着 `needs_audit` 为 `true`。同样，这似乎也是一个只要让指令更清晰并加上几个示例就能解决的问题。

将其与我们的成本模型关联起来，我们注意到有 1 个假阴性和 1 个假阳性，以及 7 个真阳性和 11 个真阴性。按照生产中观察到的频率推算，这会使我们的总成本每年增加 $63,000。

我们来修改提示词并重新运行评估，看看效果如何。我们将在指令中以一个关于机油的具体示例的形式提供更多指导（与除雪刷不同，但需要同样的推理），并从训练集（`data/train`）中选取三个示例作为少样本（few-shot）指导。

In [ ]:
first_ai_system_cost = calculate_costs(
    fp_rate=1 / 12, fn_rate=1 / 8, per_receipt_cost=0.01
)

print(f"First version of our system, estimated cost: ${first_ai_system_cost:,.0f}")

In [ ]:
nursery_receipt_details = ReceiptDetails(
    merchant="WESTERN SIERRA NURSERY",
    location=Location(city="Oakhurst", state="CA", zipcode="93644"),
    time="2024-09-27T12:33:38",
    items=[
        LineItem(
            description="Plantskydd Repellent RTU 1 Liter",
            product_code=None,
            category="Garden/Pest Control",
            item_price="24.99",
            sale_price=None,
            quantity="1",
            total="24.99",
        )
    ],
    subtotal="24.99",
    tax="1.94",
    total="26.93",
    handwritten_notes=[],
)

nursery_audit_decision = AuditDecision(
    not_travel_related=True,
    amount_over_limit=False,
    math_error=False,
    handwritten_x=False,
    reasoning="""
    1. The merchant is a plant nursery and the item purchased an insecticide, so this
       purchase is not travel-related (criterion 1 violated).
    2. The total is $26.93, under $50, so criterion 2 is not violated.
    3. The line items (1 * $24.99 + $1.94 tax) sum to $26.93, so criterion 3 is not
       violated.
    4. There are no handwritten notes or 'X's, so criterion 4 is not violated.
    Since NOT_TRAVEL_RELATED is true, the receipt must be audited.
    """,
    needs_audit=True,
)

flying_j_details = ReceiptDetails(
    merchant="Flying J #616",
    location=Location(city="Frazier Park", state="CA", zipcode=None),
    time="2024-10-01T13:23:00",
    items=[
        LineItem(
            description="Unleaded",
            product_code=None,
            category="Fuel",
            item_price="4.459",
            sale_price=None,
            quantity="11.076",
            total="49.39",
        )
    ],
    subtotal="49.39",
    tax=None,
    total="49.39",
    handwritten_notes=["yos -> home sequoia", "236660"],
)
flying_j_audit_decision = AuditDecision(
    not_travel_related=False,
    amount_over_limit=False,
    math_error=False,
    handwritten_x=False,
    reasoning="""
    1. The only item purchased is Unleaded gasoline, which is travel-related so
       NOT_TRAVEL_RELATED is false.
    2. The total is $49.39, which is under $50, so AMOUNT_OVER_LIMIT is false.
    3. The line items ($4.459 * 11.076 = $49.387884) sum to the total of $49.39, so
       MATH_ERROR is false.
    4. There is no "X" in the handwritten notes, so HANDWRITTEN_X is false.
    Since none of the criteria are violated, the receipt does not need auditing.
    """,
    needs_audit=False,
)

engine_oil_details = ReceiptDetails(
    merchant="O'Reilly Auto Parts",
    location=Location(city="Sylmar", state="CA", zipcode="91342"),
    time="2024-04-26T8:43:11",
    items=[
        LineItem(
            description="VAL 5W-20",
            product_code=None,
            category="Auto",
            item_price="12.28",
            sale_price=None,
            quantity="1",
            total="12.28",
        )
    ],
    subtotal="12.28",
    tax="1.07",
    total="13.35",
    handwritten_notes=["vista -> yos"],
)
engine_oil_audit_decision = AuditDecision(
    not_travel_related=False,
    amount_over_limit=False,
    math_error=False,
    handwritten_x=False,
    reasoning="""
    1. The only item purchased is engine oil, which might be required for a vehicle
       while traveling, so NOT_TRAVEL_RELATED is false.
    2. The total is $13.35, which is under $50, so AMOUNT_OVER_LIMIT is false.
    3. The line items ($12.28 + $1.07 tax) sum to the total of $13.35, so
       MATH_ERROR is false.
    4. There is no "X" in the handwritten notes, so HANDWRITTEN_X is false.
    None of the criteria are violated so the receipt does not need to be audited.
    """,
    needs_audit=False,
)

examples = [
    {"input": nursery_receipt_details, "output": nursery_audit_decision},
    {"input": flying_j_details, "output": flying_j_audit_decision},
    {"input": engine_oil_details, "output": engine_oil_audit_decision},
]

# Format the examples as JSON, with each example wrapped in XML tags.
example_format = """
<example>
    <input>
        {input}
    </input>
    <output>
        {output}
    </output>
</example>
"""

examples_string = ""
for example in examples:
    example_input = example["input"].model_dump_json()
    correct_output = example["output"].model_dump_json()
    examples_string += example_format.format(input=example_input, output=correct_output)

audit_prompt = f"""
Evaluate this receipt data to determine if it need to be audited based on the following
criteria:

1. NOT_TRAVEL_RELATED:
   - IMPORTANT: For this criterion, travel-related expenses include but are not limited
   to: gas, hotel, airfare, or car rental.
   - If the receipt IS for a travel-related expense, set this to FALSE.
   - If the receipt is NOT for a travel-related expense (like office supplies), set this
   to TRUE.
   - In other words, if the receipt shows FUEL/GAS, this would be FALSE because gas IS
   travel-related.
   - Travel-related expenses include anything that could be reasonably required for
   business-related travel activities. For instance, an employee using a personal
   vehicle might need to change their oil; if the receipt is for an oil change or the
   purchase of oil from an auto parts store, this would be acceptable and counts as a
   travel-related expense.

2. AMOUNT_OVER_LIMIT: The total amount exceeds $50

3. MATH_ERROR: The math for computing the total doesn't add up (line items don't sum to
   total)
   - Add up the price and quantity of each line item to get the subtotal
   - Add tax to the subtotal to get the total
   - If the total doesn't match the amount on the receipt, this is a math error
   - If the total is off by no more than $0.01, this is NOT a math error

4. HANDWRITTEN_X: There is an "X" in the handwritten notes

For each criterion, determine if it is violated (true) or not (false). Provide your
reasoning for each decision, and make a final determination on whether the receipt needs
auditing. A receipt needs auditing if ANY of the criteria are violated.

Note that violation of a criterion means that it is `true`. If any of the above four
values are `true`, then the receipt needs auditing (`needs_audit` should be `true`: it
functions as a boolean OR over all four criteria).

If the receipt contains non-travel expenses, then NOT_TRAVEL_RELATED should be `true`
and therefore NEEDS_AUDIT must also be set to `true`. IF THE RECEIPT LISTS ITEMS THAT
ARE NOT TRAVEL-RELATED, THEN IT MUST BE AUDITED. Here are some example inputs to
demonstrate how you should act:

<examples>
{examples_string}
</examples>

Return a structured response with your evaluation.
"""

我们对上面提示词所做的修改如下：

1. 在关于差旅相关费用的第 1 项下，我们添加了一个要点

```
- Travel-related expenses include anything that could be reasonably required for
  business-related travel activities. For instance, an employee using a personal
  vehicle might need to change their oil; if the receipt is for an oil change or the
  purchase of oil from an auto parts store, this would be acceptable and counts as a
  travel-related expense.
```

2. 我们添加了关于如何判断计算错误的更具规定性的指导。
   具体来说，我们添加了以下要点：

```
   - Add up the price and quantity of each line item to get the subtotal
   - Add tax to the subtotal to get the total
   - If the total doesn't match the amount on the receipt, this is a math error
   - If the total is off by no more than $0.01, this is NOT a math error
```

   这其实与我们提到的问题无关，而是我们注意到的审计模型推理中的另一个缺陷。

3. 我们添加了非常强烈的指导（实际上我们需要反复强调说明），指出非差旅相关的费用应当被审计。

```
Note that violation of a criterion means that it is `true`. If any of the above four
values are `true`, then the receipt needs auditing (`needs_audit` should be `true`: it
functions as a boolean OR over all four criteria).

If the receipt contains non-travel expenses, then NOT_TRAVEL_RELATED should be `true`
and therefore NEEDS_AUDIT must also be set to `true`. IF THE RECEIPT LISTS ITEMS THAT
ARE NOT TRAVEL-RELATED, THEN IT MUST BE AUDITED.
```

4. 我们添加了三个示例，即用 XML 标签包裹的 JSON 输入/输出对。

完成提示词修订后，我们将重新生成待评估的数据，并重新运行同样的评估来比较结果：

In [ ]:
file_content = await create_dataset_content(receipt_image_dir)

eval_run = await client.evals.runs.create(
    name="updated-receipt-processing-run",
    eval_id=full_eval.id,
    data_source={
        "type": "jsonl",
        "source": {"type": "file_content", "content": file_content},
    },
)

eval_run.report_url

再次运行评估时，我们实际上仍然有两个审计决策出错。深入研究出错的示例后发现，我们已经彻底修复了之前识别出的问题，但示例改进了推理步骤，从而暴露出另外两个问题。具体来说：

1. 有一张收据之所以需要审计，仅仅是因为提取出了错，没有识别出一个手写的“X”。审计模型的推理是正确的，但基于的是错误的数据。
2. 有一张收据的提取结果中看不到一笔 $0.35 的借记卡手续费，因此审计模型判定存在计算错误。几乎可以肯定，这是因为我们提供了更详细的指令和清晰的示例，表明它需要真正把所有明细项加起来才能判断是否存在计算错误。同样，这表明审计模型的行为是正确的，也说明我们需要纠正提取模型。

这很好，我们会在发现问题时持续迭代解决。这就是改进的循环！

### 模型选择

在项目开始时，我们通常会从能力最强的可用模型之一（例如 `o4-mini`）入手，以建立性能基线。一旦确信模型有能力解决该任务，下一步就是探索更小、更快或更具成本效益的替代方案。

优化推理成本和延迟至关重要，尤其是对于生产系统或面向客户的系统，这些因素会显著影响总体开销和用户体验。例如，从 `o4-mini` 切换到 `gpt-4.1-mini` 可以将推理成本降低近三分之二——这正是深思熟虑的模型选择带来可观节省的一个例子。

在下一节中，我们将在提取和审计两个步骤中都使用 `gpt-4.1-mini` 重新运行评估，看看更高效的模型表现如何。

In [ ]:
file_content = await create_dataset_content(receipt_image_dir, model="gpt-4.1-mini")

eval_run = await client.evals.runs.create(
    name="receipt-processing-run-gpt-4-1-mini",
    eval_id=full_eval.id,
    data_source={
        "type": "jsonl",
        "source": {"type": "file_content", "content": file_content},
    },
)

eval_run.report_url

结果相当令人鼓舞。提取准确率看起来完全没有下降。我们看到了一处回归（又是除雪刷），但审计决策的正确次数是修改提示词之前的两倍。

![评估变体](../../../images/partner_eval_variations.png)

这有力地证明了我们能够切换到更便宜的模型，但这可能需要更多的提示词工程、微调或某种形式的模型蒸馏。不过请注意，根据我们当前的模型，这已经能为我们省钱了。我们暂时还不完全相信这一点，因为样本量还不够大——我们真实的假阴性率会高于这里看到的 0。

In [ ]:
system_cost_4_1_mini = calculate_costs(
    fp_rate=1 / 12, fn_rate=0, per_receipt_cost=0.003
)

print(f"Cost using gpt-4.1-mini: ${system_cost_4_1_mini:,.0f}")


### 进一步改进

本 cookbook 关注的是评估的理念和实践，而不是全部的模型改进技术。若要提升或保持模型性能（尤其是在迁移到更小、更快或更便宜的模型时），请按顺序考虑以下步骤——从最上面开始，只在必要时才往下进行。例如，务必先优化提示词，再考虑微调；基于糟糕的提示词进行微调，可能会把糟糕的性能固化下来，即使你之后改进了提示词也无济于事。

![模型改进瀑布](../../../images/partner_model_improvement_waterfall.png)

1. **模型选择：** 尝试更聪明的模型，或增加其推理预算。
2. **提示词调优：** 澄清指令，并提供非常明确的规则。
3. **示例和上下文：** 添加少样本或多样本示例，或为问题提供更多上下文。RAG 就属于这一类，可用于动态选择相似的示例。
4. **工具使用：** 提供解决特定问题的工具，包括访问外部 API、查询数据库的能力，或以其他方式让模型能够自行获得问题的答案。
5. **辅助模型：** 添加模型来执行有限的子任务、进行监督并提供护栏，或者使用专家混合（mixture of experts），汇总多个子模型的解决方案。
6. **微调：** 使用标注的训练数据进行监督微调，使用评估评分器进行强化微调，或使用不同的输出进行直接偏好优化。

上述选项都是最大化性能的工具。当你尝试优化性价比时，通常已经完成了上述所有步骤，大多数步骤可能无需重复，但你仍然可以微调更小的模型，或使用最好的模型来训练更小的模型（模型蒸馏）。

> OpenAI Evals 的一个非常出色之处在于，你可以将同样的评分器用于[强化微调（Reinforcement Fine-Tuning）](https://cookbook.openai.com/examples/reinforcement_fine_tuning)，以极高的样本效率获得更好的模型性能。需要提醒的一点是，请确保使用单独的训练数据，不要在 RFT 过程中泄露评估数据集。

## 部署与开发后阶段
构建和部署 LLM 应用只是开始——真正的价值来自持续改进。系统上线后，要优先进行持续监控：记录追踪、跟踪输出，并使用智能采样技术主动抽取真实用户交互进行人工审核。

生产数据是演进评估数据集和训练数据集最真实的来源。定期从实际用例中收集并整理新样本，以发现差距、边界情况和新的改进机会。

在实践中，利用这些数据进行快速迭代。自动化定期微调流水线，用近期的高质量样本重新训练模型，并在新版本于评估中胜过现有版本时自动部署。收集用户的纠正和反馈，然后系统地将这些洞察反馈到提示词或再训练流程中——尤其是当它们揭示出持续存在的问题时。

通过将这些反馈循环嵌入开发后的工作流，你可以确保 LLM 应用持续适应、保持稳健，并随着用户需求的变化始终与之紧密契合。

### 贡献者
本 cookbook 由 OpenAI 与 [Fractional](https://www.fractional.ai/) 共同合作完成。

- Hugh Wimberly
- Joshua Marker
- Eddie Siegel
- Shikhar Kwatra